In [1]:
# 네이버 검색 API 예제 - 블로그 검색
import os
import urllib.request
from dotenv import load_dotenv
import pandas as pd
import json
import urllib.parse

load_dotenv()
client_id = os.getenv("client_id")
client_secret = os.getenv("client_secret")
if not client_id or not client_secret:
    raise ValueError("NAVER_CLIENT_ID / NAVER_CLIENT_SECRET 환경변수가 없습니다. (.env 확인)")

display_count=100
num_data=1000
sort='date'

encText = urllib.parse.quote("서울시 부동산")
results=[]
for idx in range(1,num_data+1,display_count):
    url = "https://openapi.naver.com/v1/search/news?query=" + encText \
         +f"&start={idx}&display={display_count}&sort={sort}" # JSON 결과
# url = "https://openapi.naver.com/v1/search/blog.xml?query=" + encText # XML 결과
    request = urllib.request.Request(url)
    request.add_header("X-Naver-Client-Id",client_id)
    request.add_header("X-Naver-Client-Secret",client_secret)
    response = urllib.request.urlopen(request)
    rescode = response.getcode()
    if(rescode==200):
        response_body = response.read()
        response_dict=json.loads(response_body.decode('utf-8'))
        results=results+response_dict['items']
    else:
        print("Error Code:", rescode)
print(f'총 데이터 개수:{len(results)}')


총 데이터 개수:1000


In [15]:
results[:1]

[{'title': '정원오 <b>서울시</b>장 출마 선언 임박? “서울을 글로벌 G2 메가시티로 만들...',
  'originallink': 'https://slownews.kr/150852',
  'link': 'https://slownews.kr/150852',
  'description': '정원오가 생각하는 ‘서울의 비전’은?<b>서울시</b>장에 출마한다면 공약은?구청장 11년 동안 기억에 남는 사건을... <b>부동산</b>도 많이 올랐다. 아파트 매매 지수가 2014년 대비 두 배가 넘는다. 상승률 기준으로 4위다. 윤석열 3년... ',
  'pubDate': 'Sun, 14 Dec 2025 21:40:00 +0900'}]

In [17]:
from datetime import datetime
import re

df=pd.DataFrame()
remove_tags=re.compile(r'<.*?>')#HTML 태그 제거를 위한 정규표현식

for item in results:
    new_data=pd.DataFrame(
        data={
            'pubDate':datetime.strptime(item['pubDate'],"%a, %d %b %Y %H:%M:%S +0900"),
            'title':re.sub(remove_tags, '',item['title']),
            'description':re.sub(remove_tags, '',item['description'])
        },
        index=[0]
    )
    df=pd.concat([df,new_data],ignore_index=True)
df.head()

,pubDate,title,description
0,2025-12-14 21:40:00,정원오 서울시장 출마 선언 임박? “서울을 글로벌 G2 메가시티로 만들...,정원오가 생각하는 ‘서울의 비전’은?서울시장에 출마한다면 공약은?구청장 11년 동안...
1,2025-12-14 21:14:00,[스트레이트] 누구를 위한 사업인가,[이광수/부동산 전문가] &quot;기업이 그렇게 (땅을) 사는 건 흔치 않은 경우...
2,2025-12-14 19:38:00,"오세훈 &quot;정부, 10.15 대책 부작용 외면…규제 완화하라&quot;",&quot;실수요자 투기꾼 취급하는 대출 정책도 즉각 전환해야&quot; 오세훈 서...
3,2025-12-14 19:32:00,오세훈 &quot;10.15 부동산대책은 악정(惡政)&quot;,"&quot;정부, 주거 불안 해소는커녕 오히려 부추긴 꼴&quot; 오세훈 서울시장..."
4,2025-12-14 18:46:00,"[EBN 오늘(14일) 이슈 종합] 재계, 연말 인사 마무리 후 'AI 중심' 내년...",14일 업계에 따르면 주요 부동산 연구기관들은 내년 전국 집값을 보합 내지 소폭 하...


In [18]:
df.to_csv('naver_news.csv', index=False, encoding='utf-8')